# Intelligent Bookshop Chatbot Using NLP

**Project:** Design and Develop an Intelligent Bookshop Chatbot Using NLP  
**Tools:** Python, NLTK, NumPy, Keras/TensorFlow, JSON intents dataset, CSV book data

---
## Project Overview
This chatbot can:
- Recommend books by genre
- Answer customer questions
- Handle greetings and farewells
- Provide category-based book suggestions
- Respond to different user intents with confidence threshold handling

## Setup & Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import random
import pickle
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

import matplotlib.pyplot as plt

lemmatizer = WordNetLemmatizer()
print('All libraries imported successfully!')

---
## Task 1: Data Preprocessing

Steps:
1. Load the JSON dataset
2. Tokenize patterns
3. Apply lemmatization
4. Remove punctuation
5. Build: words vocabulary, classes list, documents list

In [ ]:
# ── 1. Load the JSON dataset ──────────────────────────────────────────────────
with open('intents (1).json', encoding='utf-8') as f:
    intents = json.load(f)

print(f'Loaded intents file successfully.')
print(f'Number of intents (tags): {len(intents["intents"])}')
print(f'Tags: {[i["tag"] for i in intents["intents"]]}')

In [ ]:
# ── 2-5. Tokenize, Lemmatize, Remove Punctuation, Build vocabulary ─────────────
words = []       # vocabulary of all unique lemmatized words
classes = []     # list of all intent tags
documents = []   # list of (token_list, tag) tuples
ignore_chars = ['?', '!', '.', ',', "'", '"', ';', ':']

for intent in intents['intents']:
    tag = intent['tag']

    # Collect unique class labels
    if tag not in classes:
        classes.append(tag)

    for pattern in intent['patterns']:
        # 2. Tokenize each pattern
        token_list = word_tokenize(pattern)

        # 3 & 4. Lemmatize and remove punctuation
        token_list = [
            lemmatizer.lemmatize(w.lower())
            for w in token_list
            if w not in ignore_chars
        ]

        words.extend(token_list)
        documents.append((token_list, tag))

# 5. Build final sorted vocabulary (unique words)
words = sorted(set(words))
classes = sorted(set(classes))

print(f'Total unique words in vocabulary : {len(words)}')
print(f'Total intent classes             : {len(classes)}')
print(f'Total documents (pattern-tag)    : {len(documents)}')
print(f'\nSample vocabulary (first 20): {words[:20]}')
print(f'\nSample documents (first 3):')
for doc in documents[:3]:
    print(f'  tokens={doc[0]}  tag="{doc[1]}"')

In [ ]:
# Save preprocessed data for later use
pickle.dump(words,   open('words.pkl',   'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))
print('Saved words.pkl and classes.pkl')

---
## Task 2: Feature Engineering

Steps:
1. Implement Bag-of-Words (BoW)
2. Create training matrix
3. Convert labels to one-hot encoding
4. Prepare `train_x` and `train_y`

In [ ]:
# ── 1 & 2. Bag-of-Words → Training matrix ────────────────────────────────────
training = []
output_empty = [0] * len(classes)  # template zero vector for one-hot labels

for token_list, tag in documents:
    # BoW: 1 if word in vocabulary appears in this pattern, else 0
    bag = [1 if w in token_list else 0 for w in words]

    # 3. One-hot encode the label
    output_row = list(output_empty)
    output_row[classes.index(tag)] = 1

    training.append([bag, output_row])

# Shuffle and split into features / labels
random.shuffle(training)
training = np.array(training, dtype=object)

# 4. Prepare train_x and train_y
train_x = np.array(list(training[:, 0]), dtype=np.float32)
train_y = np.array(list(training[:, 1]), dtype=np.float32)

print(f'train_x shape : {train_x.shape}  (samples × vocabulary size)')
print(f'train_y shape : {train_y.shape}  (samples × number of classes)')
print(f'\nExample BoW vector (first sample, first 30 dims):')
print(train_x[0, :30])
print(f'\nCorresponding one-hot label:')
print(train_y[0])

### Explanation: Bag-of-Words

**Why BoW is sparse:**  
The vocabulary built from all patterns contains many unique words. Each pattern uses only a small fraction of these words, so most entries in the BoW vector are 0. With a vocabulary of ~200+ words but patterns of only 3–8 words, the feature vectors are extremely sparse (>95% zeros).

**Limitations of BoW:**  
1. **No word order** — "book recommend" and "recommend book" produce identical vectors.  
2. **No semantics** — synonyms ("suggest" vs. "recommend") are treated as completely different words.  
3. **High dimensionality** — vector length equals vocabulary size, which grows with data.  
4. **Out-of-vocabulary (OOV)** — words not seen during training are silently ignored at inference time.  
5. **Equal weight** — frequent function words ("a", "the") are treated the same as informative words.

---
## Task 3: Neural Network Model

Architecture:  
`Dense(128, ReLU) → Dropout(0.5) → Dense(64, ReLU) → Dropout(0.5) → Dense(num_classes, Softmax)`  

Training: ≥ 200 epochs

In [ ]:
# ── Build the model ───────────────────────────────────────────────────────────
model = Sequential([
    Dense(128, input_shape=(len(train_x[0]),), activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(len(train_y[0]), activation='softmax')
])

sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

model.summary()

In [ ]:
# ── Train the model ───────────────────────────────────────────────────────────
EPOCHS = 200

history = model.fit(
    train_x, train_y,
    epochs=EPOCHS,
    batch_size=5,
    verbose=1
)

# Save the trained model
model.save('bookshop_chatbot_model.h5')
print('\nModel saved as bookshop_chatbot_model.h5')

In [ ]:
# ── Plot training accuracy and loss ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], color='steelblue', linewidth=2)
axes[0].set_title('Training Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

axes[1].plot(history.history['loss'], color='tomato', linewidth=2)
axes[1].set_title('Training Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bookshop Chatbot – Model Training', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

final_acc = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]
print(f'\nFinal training accuracy : {final_acc:.4f} ({final_acc*100:.1f}%)')
print(f'Final training loss     : {final_loss:.4f}')

---
## Task 4: Prediction Pipeline

Functions implemented:
- `clean_up_sentence()` — tokenize + lemmatize input
- `bow()` — convert sentence to Bag-of-Words vector
- `predict_class()` — predict the intent tag using the model
- `getResponse()` — select a response for the predicted intent
- `chatbot_response()` — end-to-end pipeline

In [ ]:
# ── Load saved artifacts ──────────────────────────────────────────────────────
model   = load_model('bookshop_chatbot_model.h5')
words   = pickle.load(open('words.pkl',   'rb'))
classes = pickle.load(open('classes.pkl', 'rb'))

with open('intents (1).json', encoding='utf-8') as f:
    intents = json.load(f)

# Load book CSV for dynamic recommendations
books_df = pd.read_csv('data.csv')
print('Model and data loaded.')
print(f'Vocabulary size : {len(words)} | Classes : {len(classes)}')

In [ ]:
# ── Prediction helper functions ───────────────────────────────────────────────

def clean_up_sentence(sentence):
    """Tokenize and lemmatize the input sentence."""
    tokens = word_tokenize(sentence)
    tokens = [lemmatizer.lemmatize(t.lower()) for t in tokens]
    return tokens


def bow(sentence, words):
    """Convert a sentence to a Bag-of-Words NumPy vector."""
    token_list = clean_up_sentence(sentence)
    bag = [1 if w in token_list else 0 for w in words]
    return np.array(bag, dtype=np.float32)


def predict_class(sentence, model, threshold=0.25):
    """
    Run the model and return a list of dicts sorted by probability.
    Only intents above `threshold` are returned.
    """
    bow_vec = bow(sentence, words)
    res = model.predict(bow_vec.reshape(1, -1), verbose=0)[0]

    results = [
        {'intent': classes[i], 'probability': float(res[i])}
        for i in range(len(res))
        if res[i] > threshold
    ]
    results.sort(key=lambda x: x['probability'], reverse=True)
    return results


def getResponse(intents_list, intents_json):
    """
    Select a response for the top predicted intent.
    Handles both plain-string responses and book-dict responses.
    """
    if not intents_list:
        return "I'm not sure I understand. Could you rephrase that?"

    tag = intents_list[0]['intent']

    for intent in intents_json['intents']:
        if intent['tag'] == tag:
            responses = intent['responses']

            if not responses:
                return "I don't have a response for that yet."

            chosen = random.choice(responses)

            # Book-dict response format
            if isinstance(chosen, dict):
                book_title   = chosen.get('Book', 'Unknown')
                description  = chosen.get('Feedback', '')
                rating       = chosen.get('Rate', 'N/A')
                return (
                    f"Book Recommendation: {book_title}\n"
                    f"Rating: {rating}/5\n\n"
                    f"{description}"
                )

            # Plain string response
            return chosen

    return "I'm not sure I understand. Could you rephrase that?"


def chatbot_response(user_message, threshold=0.25):
    """End-to-end pipeline: message → predicted intents → response string."""
    intents_list = predict_class(user_message, model, threshold=threshold)
    response     = getResponse(intents_list, intents)
    return response


print('Prediction pipeline ready.')

In [ ]:
# ── Test with 10 different queries ────────────────────────────────────────────
test_queries = [
    "Hello!",
    "Good morning",
    "Can you recommend a book?",
    "I want a Fiction book",
    "Recommend a book in Science fiction",
    "I'm looking for a Mystery novel",
    "Give me a book about History",
    "I love Cooking books",
    "Thank you so much!",
    "Goodbye",
    "Recommend something in Fantasy fiction",
    "I'd like a Psychology book",
]

print('=' * 65)
print('CHATBOT RESPONSE TEST  (10+ queries)'.center(65))
print('=' * 65)

for i, query in enumerate(test_queries, 1):
    response = chatbot_response(query)
    print(f'\n[{i:02d}] User : {query}')
    print(f'     Bot  : {response[:200]}...' if len(response) > 200 else f'     Bot  : {response}')
    print('-' * 65)

---
## Task 5: Improvement & Enhancement

### Option D — Confidence Threshold Handling

When the model's top predicted class has a probability **below the confidence threshold**, the chatbot declines to answer instead of giving a wrong response. This prevents hallucinated or irrelevant replies.

Additionally, for **book-category intents** we dynamically pull a **top-rated book** directly from the `data.csv` dataset, giving richer and fresher recommendations beyond what is hard-coded in the intents file.

In [ ]:
# ── Enhanced response with confidence threshold + CSV book lookup ─────────────

# Map intent tags to data.csv categories (exact or partial match)
CATEGORY_TAGS = {
    i['tag'] for i in intents['intents']
    if i['tag'] not in ('greeting', 'goodbye', 'thanks', 'book_search')
}


def get_csv_book_recommendation(category_tag, books_df, top_n=5):
    """
    Look up a random top-rated book from data.csv for the given category tag.
    Returns a formatted string, or None if no match found.
    """
    mask = books_df['categories'].str.contains(
        category_tag, case=False, na=False
    )
    subset = books_df[mask].dropna(subset=['average_rating'])

    if subset.empty:
        return None

    # Pick randomly from top-N highest-rated books
    top_books = subset.nlargest(top_n, 'average_rating')
    book = top_books.sample(1).iloc[0]

    title    = book['title']
    authors  = book.get('authors', 'Unknown')
    rating   = book.get('average_rating', 'N/A')
    desc     = str(book.get('description', '')).strip()
    desc_short = desc[:300] + '...' if len(desc) > 300 else desc

    return (
        f"[Live Recommendation from our catalogue]\n"
        f"Book    : {title}\n"
        f"Author  : {authors}\n"
        f"Rating  : {rating}/5\n\n"
        f"{desc_short}"
    )


def chatbot_response_enhanced(user_message, threshold=0.35):
    """
    Enhanced pipeline:
    1. Predict intent.
    2. If top confidence < threshold → politely decline.
    3. For book-category intents → try CSV lookup first (richer info),
       fallback to intents.json response.
    4. For all other intents → use intents.json response.
    """
    intents_list = predict_class(user_message, model, threshold=0.0)

    if not intents_list or intents_list[0]['probability'] < threshold:
        return (
            f"I'm not confident enough to answer that (confidence: "
            f"{intents_list[0]['probability']:.2%} < {threshold:.0%} threshold).\n"
            f"Could you please rephrase or ask about a specific book genre?"
        )

    top_tag         = intents_list[0]['intent']
    top_confidence  = intents_list[0]['probability']

    # Try dynamic CSV recommendation for book categories
    if top_tag in CATEGORY_TAGS:
        csv_response = get_csv_book_recommendation(top_tag, books_df)
        if csv_response:
            return f"[Confidence: {top_confidence:.1%}]\n\n" + csv_response

    # Fallback to intents.json response
    response = getResponse(intents_list, intents)
    return f"[Confidence: {top_confidence:.1%}]\n\n" + response


print('Enhanced prediction pipeline ready.')

In [ ]:
# ── Demonstrate Task 5 enhancements ──────────────────────────────────────────
enhancement_tests = [
    # Normal high-confidence queries
    ("Hello there!",                     'high confidence greeting'),
    ("Recommend a Fiction book",          'high confidence + CSV lookup'),
    ("I want a Science book",             'high confidence + CSV lookup'),
    # Low-confidence / ambiguous query — should trigger threshold handler
    ("asdfghjkl",                         'nonsense → low confidence'),
    ("Tell me something",                 'vague → may be low confidence'),
    ("I need a book about Travel",        'category intent + CSV'),
    ("Recommend Psychology books",        'category intent + CSV'),
    ("Bye!",                              'goodbye intent'),
]

print('=' * 70)
print('TASK 5: ENHANCED CHATBOT – CONFIDENCE THRESHOLD + CSV LOOKUP'.center(70))
print('=' * 70)

for query, note in enhancement_tests:
    print(f'\nTest ({note})')
    print(f'  User : {query}')
    resp = chatbot_response_enhanced(query)
    lines = resp.split('\n')
    for line in lines[:6]:          # show first 6 lines for readability
        print(f'  Bot  : {line}')
    if len(lines) > 6:
        print(f'  Bot  : ... [{len(lines)-6} more lines]')
    print('-' * 70)

---
## Chatbot UI Preview

A full standalone GUI is provided in **`chatbot_ui.py`** (run with `python chatbot_ui.py`).

Below is a minimal in-notebook interactive session using `input()`:

In [ ]:
# ── Simple in-notebook chat loop ──────────────────────────────────────────────
# Run this cell and type messages; type 'quit' or 'exit' to stop.

print('Bookshop Chatbot is running! (type "quit" to exit)\n')
print('─' * 55)

while True:
    user_input = input('You: ').strip()
    if not user_input:
        continue
    if user_input.lower() in ('quit', 'exit', 'bye'):
        print('Bot: Goodbye! Happy reading! 📚')
        break
    response = chatbot_response_enhanced(user_input)
    print(f'Bot: {response}\n')